In [8]:
import pandas as pd

reduced_df = pd.read_feather('./processed/reduced_coerced_engagement_list.feather')

In [9]:
research_and_development = reduced_df[reduced_df['Engagement Service Code'] == '10676']
initial_count = len(research_and_development)

# Apply all filtering criteria for Total Addressable Revenue
rd_initial_count = len(research_and_development)
rd_previous_count = rd_initial_count
print(f"Initial subset dataframe shape: {reduced_df.shape} ({initial_count:,} rows)")

# Filter Release Date for dates greater than 01/01/2024
research_and_development = research_and_development[research_and_development['Release Date'] > '2024-01-01']
rd_current_count = len(research_and_development)
rd_reduction_from_prev = rd_previous_count - rd_current_count
rd_pct_reduction_from_prev = (rd_reduction_from_prev / rd_previous_count * 100) if rd_previous_count > 0 else 0
print(f"After filtering Release Date > 01/01/2024: {research_and_development.shape} ({rd_current_count:,} rows, {rd_reduction_from_prev:,} rows removed, {rd_pct_reduction_from_prev:.1f}% reduction from previous step)")
rd_previous_count = rd_current_count

# Filter ANSR / Tech Revenue ETD for values greater than 0
research_and_development = research_and_development[research_and_development['ANSR / Tech Revenue ETD'] != 0]
rd_current_count = len(research_and_development)
rd_reduction_from_prev = rd_previous_count - rd_current_count
rd_pct_reduction_from_prev = (rd_reduction_from_prev / rd_previous_count * 100) if rd_previous_count > 0 else 0
print(f"After filtering ANSR / Tech Revenue ETD > 0: {research_and_development.shape} ({rd_current_count:,} rows, {rd_reduction_from_prev:,} rows removed, {rd_pct_reduction_from_prev:.1f}% reduction from previous step)")
rd_previous_count = rd_current_count

Initial subset dataframe shape: (210873, 12) (2,720 rows)
After filtering Release Date > 01/01/2024: (1407, 12) (1,407 rows, 1,313 rows removed, 48.3% reduction from previous step)
After filtering ANSR / Tech Revenue ETD > 0: (1382, 12) (1,382 rows, 25 rows removed, 1.8% reduction from previous step)


In [10]:
prodigy = pd.read_excel('./raw/prodigy/ClientEngagements_20260120_113520.xlsx')

In [7]:
# Check column names in both dataframes
print("Research & Development columns:")
print(research_and_development.columns.tolist())
print("\nProdigy columns:")
print(prodigy.columns.tolist())

Research & Development columns:
['Client ID', 'Client', 'Account Channel', 'Engagement ID', 'Engagement Status', 'Engagement Service Code', 'Release Date', 'Engagement', 'ANSR / Tech Revenue ETD', 'ANSR / Tech Revenue FYTD', 'TER ETD', 'TER FYTD']

Prodigy columns:
['Client Name', 'ClientID', 'Engagement Name', 'EngagementID', 'RowError', 'RowState', 'Table', 'ItemArray', 'HasErrors']


## Prodigy Metrics Calculation

Calculate the three key metrics:
1. Total ANSR FYTD Revenue supported by Prodigy
2. Percentage of engagements supported by Prodigy
3. Percentage of ANSR FYTD Revenue supported by Prodigy

In [11]:
# Find R&D engagements where the client appears in Prodigy
# Get unique client IDs from Prodigy
prodigy_clients = prodigy['ClientID'].unique()

# Filter R&D engagements to only those with clients in Prodigy
matched_engagements = research_and_development[
    research_and_development['Client ID'].isin(prodigy_clients)
].copy()

print(f"Number of R&D engagements with clients in Prodigy: {len(matched_engagements):,}")
print(f"Total R&D engagements (denominator): {len(research_and_development):,}")
print(f"Unique clients in Prodigy: {len(prodigy_clients):,}")
print(f"Total Prodigy engagements: {len(prodigy):,}")

Number of R&D engagements with clients in Prodigy: 1,153
Total R&D engagements (denominator): 1,382
Unique clients in Prodigy: 954
Total Prodigy engagements: 2,580


In [6]:
# Metric 1: Total ANSR FYTD Revenue supported by Prodigy
prodigy_supported_revenue = matched_engagements['ANSR / Tech Revenue FYTD'].sum()
print(f"\n{'='*60}")
print(f"Metric 1: ANSR FYTD Revenue Supported by Prodigy")
print(f"{'='*60}")
print(f"Total ANSR FYTD Revenue in Prodigy: ${prodigy_supported_revenue:,.2f}")

# Metric 2: Percentage of engagements supported by Prodigy
total_rd_engagements = len(research_and_development)
prodigy_engagement_count = len(matched_engagements)
pct_engagements_supported = (prodigy_engagement_count / total_rd_engagements * 100) if total_rd_engagements > 0 else 0

print(f"\n{'='*60}")
print(f"Metric 2: Percentage of Engagements Supported by Prodigy")
print(f"{'='*60}")
print(f"Engagements in Prodigy (numerator): {prodigy_engagement_count:,}")
print(f"Total R&D Engagements (denominator): {total_rd_engagements:,}")
print(f"Percentage: {pct_engagements_supported:.2f}%")

# Metric 3: Percentage of ANSR FYTD Revenue supported by Prodigy
total_rd_revenue = research_and_development['ANSR / Tech Revenue FYTD'].sum()
pct_revenue_supported = (prodigy_supported_revenue / total_rd_revenue * 100) if total_rd_revenue > 0 else 0

print(f"\n{'='*60}")
print(f"Metric 3: Percentage of ANSR FYTD Revenue Supported by Prodigy")
print(f"{'='*60}")
print(f"ANSR FYTD Revenue in Prodigy (numerator): ${prodigy_supported_revenue:,.2f}")
print(f"Total ANSR FYTD Revenue (denominator): ${total_rd_revenue:,.2f}")
print(f"Percentage: {pct_revenue_supported:.2f}%")


Metric 1: ANSR FYTD Revenue Supported by Prodigy
Total ANSR FYTD Revenue in Prodigy: $42,237,838.55

Metric 2: Percentage of Engagements Supported by Prodigy
Engagements in Prodigy (numerator): 1,153
Total R&D Engagements (denominator): 1,382
Percentage: 83.43%

Metric 3: Percentage of ANSR FYTD Revenue Supported by Prodigy
ANSR FYTD Revenue in Prodigy (numerator): $42,237,838.55
Total ANSR FYTD Revenue (denominator): $49,066,895.31
Percentage: 86.08%


In [7]:
# Create engagement-level output for Excel
engagement_output = research_and_development.copy()

# Add flag for whether client is in Prodigy
engagement_output['Client_in_Prodigy'] = engagement_output['Client ID'].isin(prodigy_clients)

# Add count of Prodigy engagements for each client
prodigy_count_by_client = prodigy.groupby('ClientID').size().reset_index(name='Prodigy_Engagement_Count')
engagement_output = engagement_output.merge(
    prodigy_count_by_client,
    left_on='Client ID',
    right_on='ClientID',
    how='left'
)
engagement_output['Prodigy_Engagement_Count'] = engagement_output['Prodigy_Engagement_Count'].fillna(0).astype(int)
engagement_output = engagement_output.drop('ClientID', axis=1)

# Reorder columns for better readability
cols = ['Client ID', 'Client', 'Engagement ID', 'Engagement', 'Engagement Status', 
        'Client_in_Prodigy', 'Prodigy_Engagement_Count',
        'ANSR / Tech Revenue FYTD', 'ANSR / Tech Revenue ETD', 
        'TER FYTD', 'TER ETD', 'Release Date',
        'Account Channel', 'Engagement Service Code']
engagement_output = engagement_output[cols]

# Sort by whether in Prodigy (True first) and then by revenue descending
engagement_output = engagement_output.sort_values(
    by=['Client_in_Prodigy', 'ANSR / Tech Revenue FYTD'], 
    ascending=[False, False]
)

# Export to Excel
output_file = './processed/prodigy_engagement_analysis.xlsx'
engagement_output.to_excel(output_file, index=False, sheet_name='R&D_Prodigy_Analysis')

print(f"Engagement-level analysis exported to: {output_file}")
print(f"Total rows: {len(engagement_output):,}")
print(f"Engagements with clients in Prodigy: {engagement_output['Client_in_Prodigy'].sum():,}")
print(f"Engagements without clients in Prodigy: {(~engagement_output['Client_in_Prodigy']).sum():,}")

Engagement-level analysis exported to: ./processed/prodigy_engagement_analysis.xlsx
Total rows: 1,382
Engagements with clients in Prodigy: 1,153
Engagements without clients in Prodigy: 229


In [12]:
# Diagnostic: Check for duplicates in the data sources
print("=== DIAGNOSTIC: Checking for Duplicates ===")
print(f"\nOriginal research_and_development shape: {research_and_development.shape}")
print(f"Unique Engagement IDs in R&D: {research_and_development['Engagement ID'].nunique():,}")
print(f"Duplicate Engagement IDs in R&D: {research_and_development['Engagement ID'].duplicated().sum():,}")

print(f"\nProdigy shape: {prodigy.shape}")
print(f"Unique ClientIDs in Prodigy: {prodigy['ClientID'].nunique():,}")
print(f"Unique EngagementIDs in Prodigy: {prodigy['EngagementID'].nunique():,}")

# Check if there are duplicate Client IDs in research_and_development
print(f"\nUnique Client IDs in R&D: {research_and_development['Client ID'].nunique():,}")
print(f"Duplicate Client IDs in R&D: {research_and_development['Client ID'].duplicated().sum():,}")

=== DIAGNOSTIC: Checking for Duplicates ===

Original research_and_development shape: (1382, 12)
Unique Engagement IDs in R&D: 1,382
Duplicate Engagement IDs in R&D: 0

Prodigy shape: (2580, 9)
Unique ClientIDs in Prodigy: 953
Unique EngagementIDs in Prodigy: 2,579

Unique Client IDs in R&D: 798
Duplicate Client IDs in R&D: 583


In [13]:
# Verify: Re-read the Excel file to check actual row count
import openpyxl
wb = openpyxl.load_workbook('./processed/prodigy_engagement_analysis.xlsx')
ws = wb.active
excel_row_count = ws.max_row - 1  # Subtract 1 for header row
print(f"\n=== Excel File Verification ===")
print(f"Rows in Excel file (excluding header): {excel_row_count:,}")
print(f"Expected rows (from research_and_development): {len(research_and_development):,}")
print(f"Match: {excel_row_count == len(research_and_development)}")

# Check for duplicates in the engagement_output before export
print(f"\n=== Checking engagement_output DataFrame ===")
print(f"Total rows in engagement_output: {len(engagement_output):,}")
print(f"Unique Engagement IDs in engagement_output: {engagement_output['Engagement ID'].nunique():,}")
print(f"Duplicate Engagement IDs in engagement_output: {engagement_output['Engagement ID'].duplicated().sum():,}")


=== Excel File Verification ===
Rows in Excel file (excluding header): 1,382
Expected rows (from research_and_development): 1,382
Match: True

=== Checking engagement_output DataFrame ===
Total rows in engagement_output: 1,382
Unique Engagement IDs in engagement_output: 1,382
Duplicate Engagement IDs in engagement_output: 0


### Summary: Understanding Client vs. Engagement Counts

**Key Points:**
- **1,382 R&D Engagements** (unique engagement records)
- **798 Unique Clients** across those engagements
- Some clients have multiple R&D engagements (this is normal and expected)

**Excel File Contents:**
- The Excel file has 1,382 rows (one per engagement)
- No duplicate engagements
- The `Client_in_Prodigy` flag shows whether that engagement's client appears in Prodigy
- Multiple engagements can share the same client